# Notebook 15 — actions-v2 έναντι actions-v3 στο ίδιο Qwen 1.5B

Συγκρίνει την προηγούμενη παραλλαγή **C** με τη νέα **E**, αλλάζοντας μόνο τις οδηγίες
του system prompt. Κρατά τα ίδια δύο παραδείγματα, τα tool messages, το revision του
Qwen 1.5B, το greedy decoding, τα 400 νέα tokens, το routing και τον parser.

Πριν από το **Run all**: επίλεξε Colab GPU. Όταν ζητηθεί, ανέβασε το ZIP του
προηγούμενου generator development run (`20261003T084949Z_51c7a13e`).
Χρησιμοποιούνται ακριβώς οι αποθηκευμένες προβλέψεις DistilBERT/Jev για τις ίδιες
24 επινοημένες κριτικές ανάπτυξης. **Δεν γίνονται νέες κλήσεις Jev και δεν χρειάζεται API key.**
Δεν διαβάζεται ούτε εκτελείται το reserved holdout. Δεν γίνεται εκπαίδευση ή δημοσίευση.

| Παραλλαγή | Μοντέλο | Prompt |
|---|---|---|
| C | Qwen2.5-1.5B-Instruct, προηγούμενο revision | προηγούμενο actions-v2 |
| E | ακριβώς τα ίδια βάρη | actions-v3: πραγματικά εκκρεμή προβλήματα, μέτρα για τη διοίκηση |

Όλες οι κριτικές ερωτώνται για διάγνωση του generator, ακόμη και όταν το παραγωγικό
routing θα τις παρέλειπε. Οι μετρήσεις δομής δεν είναι ακρίβεια ή χρησιμότητα.
Στο τέλος κατεβαίνει νέο ZIP με 48 αποτελέσματα και κενό φύλλο ανθρώπινης αξιολόγησης.

## 1. Καρφωμένος κώδικας και προηγούμενο run

In [ ]:
from pathlib import Path

SOURCE_COMMIT = "7aa6bf14afb419add1953afa256ccc2a18a9c3f1"
SCRIPT_SHA256 = "e353897b043f583f05967c8618abf6f6b0358ba56c045b83b49638ecefa37ce5"
REVIEWS_SHA256 = "69718a4170ca151fe880455f17b243b79bc7eaa04ce4a9eea8048ef8f83ec6f8"
REFERENCE_RUN_SHA256 = "edd3d837e3a3299452b60ed2a7af1ea71be647afba036730ba28a852b2068312"
REFERENCE_UPSTREAM_SHA256 = "3d57b39b8350383beedb4f708eaf8b78b91196a16a2757fb52b923eefa728235"
MODEL_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
REPO_DIR = Path("/content/hotel-review-prompt-v3")
SECRET_NAME = "OPENROUTER_API_KEY"
EXPECTED_CANDIDATES = ("C", "E")
# Set to an existing local ZIP path, or leave None to select a file when prompted.
REFERENCE_ZIP = None
REFERENCE_READY = SETUP_READY = GPU_READY = False
print("Κώδικας:", SOURCE_COMMIT)
print("24 development κριτικές, δύο prompts, ίδιο Qwen 1.5B. Νέες κλήσεις Jev: 0.")

## 2. Ανέβασμα του προηγούμενου ZIP

Διαβάζει μόνο `run.json` και `upstream.json`, ελέγχει τα γνωστά SHA-256 και απαιτεί
ολοκληρωμένο development run. Δεν εκτελεί ούτε εξάγει κώδικα από το ZIP.

In [ ]:
import hashlib
import json
import os
import zipfile
from google.colab import files

REFERENCE_READY = False
if REFERENCE_ZIP is None:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("Επίλεξε μόνο το ZIP του προηγούμενου generator development run.")
    REFERENCE_ZIP = Path(next(iter(uploaded)))
    del uploaded
else:
    REFERENCE_ZIP = Path(REFERENCE_ZIP)

reference_bytes = {}
with zipfile.ZipFile(REFERENCE_ZIP) as archive:
    members = archive.infolist()
    for name in ("run.json", "upstream.json"):
        matches = [item for item in members if item.filename == name and not item.is_dir()]
        if len(matches) != 1 or matches[0].file_size > 8_000_000:
            raise RuntimeError("Το ZIP πρέπει να περιέχει ακριβώς ένα μικρό " + name)
        reference_bytes[name] = archive.read(matches[0])
if hashlib.sha256(reference_bytes["run.json"]).hexdigest() != REFERENCE_RUN_SHA256:
    raise RuntimeError("Δεν είναι το αναμενόμενο προηγούμενο run.json.")
if hashlib.sha256(reference_bytes["upstream.json"]).hexdigest() != REFERENCE_UPSTREAM_SHA256:
    raise RuntimeError("Οι αποθηκευμένες προβλέψεις έχουν διαφορετικό SHA-256.")
reference_info = json.loads(reference_bytes["run.json"])
if (reference_info.get("split") != "dev" or reference_info.get("execution_complete") is not True
        or reference_info.get("dataset_sha256") != REVIEWS_SHA256
        or reference_info["candidates"]["C"]["revision"] != MODEL_REVISION):
    raise RuntimeError("Το προηγούμενο run δεν έχει τα αναμενόμενα δεδομένα, βάρη ή κατάσταση.")
REFERENCE_DIR = REPO_DIR.parent / ("generator_reference_" + REFERENCE_RUN_SHA256[:12])
REFERENCE_DIR.mkdir(parents=True, exist_ok=True)
for name, content in reference_bytes.items():
    target = REFERENCE_DIR / name
    if target.exists() and target.read_bytes() != content:
        raise RuntimeError("Υπάρχει διαφορετικό τοπικό αρχείο αναφοράς: " + name)
    target.write_bytes(content)
del reference_bytes
REFERENCE_READY = True
print("Προηγούμενο run και upstream: OK. Δεν χρειάζεται OpenRouter Secret.")

## 3. Προετοιμασία

Χρησιμοποιεί δικό του checkout και το GPU PyTorch του Colab. Αφαιρεί τα αχρησιμοποίητα
gradio/gradio-client/diffusers που συγκρούονταν με το Hub και καρφώνει τις ίδιες
εκδόσεις transformers, huggingface-hub και accelerate με το προηγούμενο πείραμα.

In [ ]:
import hashlib
import importlib.metadata as metadata
import json
import subprocess
import sys

SETUP_READY = False
GPU_READY = False
if not REFERENCE_READY:
    raise RuntimeError("Ολοκλήρωσε πρώτα τον έλεγχο του προηγούμενου ZIP.")

setup_env = {name: value for name, value in os.environ.items() if name != SECRET_NAME}

def setup_command(arguments, *, cwd=None, capture=False):
    completed = subprocess.run(
        arguments, cwd=cwd, env=setup_env, check=True, text=True,
        stdout=subprocess.PIPE if capture else None,
    )
    return completed.stdout.strip() if capture else None

if not REPO_DIR.exists():
    setup_command([
        "git", "clone", "--quiet",
        "https://github.com/krimits/hotel-review-nlp.git", str(REPO_DIR),
    ])
    setup_command(["git", "checkout", "--quiet", "--detach", SOURCE_COMMIT], cwd=REPO_DIR)
current_commit = setup_command(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture=True)
if current_commit != SOURCE_COMMIT:
    raise RuntimeError("Το checkout έχει διαφορετικό commit. Χρησιμοποίησε νέα συνεδρία Colab.")
if setup_command(["git", "status", "--porcelain", "--untracked-files=no"],
                 cwd=REPO_DIR, capture=True):
    raise RuntimeError("Υπάρχουν αλλαγές στα αρχεία του checkout. Δεν γίνεται αυτόματο reset.")

def file_sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

SCRIPT_PATH = REPO_DIR / "scripts/compare_triage_generators.py"
REVIEWS_PATH = REPO_DIR / "docs/experiments/triage_generator/dev.json"
if file_sha256(SCRIPT_PATH) != SCRIPT_SHA256 or file_sha256(REVIEWS_PATH) != REVIEWS_SHA256:
    raise RuntimeError("Τα fingerprints του script ή των κριτικών δεν συμφωνούν.")
fixture_reviews = json.loads(REVIEWS_PATH.read_text(encoding="utf-8"))["reviews"]
EXPECTED_IDS = {review["id"] for review in fixture_reviews}
if len(fixture_reviews) != 24 or len(EXPECTED_IDS) != 24:
    raise RuntimeError("Το fixture πρέπει να έχει 24 διαφορετικά IDs.")

# Reuse Colab's GPU-enabled torch. Do not download or upgrade torch here.
try:
    metadata.version("torch")
except metadata.PackageNotFoundError:
    raise RuntimeError("Δεν υπάρχει PyTorch. Άνοιξε το notebook σε συνεδρία Colab με GPU.") from None

unused_packages = []
for package in ("gradio", "gradio-client", "diffusers"):
    try:
        metadata.version(package)
        unused_packages.append(package)
    except metadata.PackageNotFoundError:
        pass
pip = [sys.executable, "-m", "pip", "--disable-pip-version-check"]
if unused_packages:
    setup_command(pip + ["uninstall", "--quiet", "-y", *unused_packages])
requirements = [
    "transformers==4.56.2", "huggingface-hub==0.36.2", "accelerate==1.10.1",
    "fastapi>=0.110", "httpx>=0.27",
]
core_requirements = {
    "numpy": "numpy>=1.26", "pandas": "pandas>=2.1", "pyarrow": "pyarrow>=14.0",
    "joblib": "joblib>=1.3", "scikit-learn": "scikit-learn>=1.4", "scipy": "scipy>=1.11",
    "pyyaml": "pyyaml>=6.0", "tqdm": "tqdm>=4.66", "matplotlib": "matplotlib>=3.8",
    "seaborn": "seaborn>=0.13",
}
for package, requirement in core_requirements.items():
    try:
        metadata.version(package)
    except metadata.PackageNotFoundError:
        requirements.append(requirement)
setup_command(pip + ["install", "--quiet", *requirements])
setup_command(pip + ["install", "--quiet", "--no-deps", "-e", str(REPO_DIR)])
SETUP_READY = True
print("Checkout και fingerprints: OK.")

## 4. GPU και έλεγχος imports σε νέο Python process

In [ ]:
import importlib.metadata as metadata
import textwrap

GPU_READY = False
if not SETUP_READY:
    raise RuntimeError("Ολοκλήρωσε πρώτα το κελί προετοιμασίας.")
environment_code = """
import json
import torch
import transformers
import huggingface_hub
from reviewnlp.triage.pipeline import TriagePipeline
from reviewnlp.triage.qwen_generator import QwenActionGenerator

if not torch.cuda.is_available():
    raise RuntimeError("Δεν υπάρχει GPU. Colab: Runtime -> Change runtime type -> GPU.")
assert transformers.__version__ == "4.56.2", "Διαφορετική έκδοση transformers"
assert huggingface_hub.__version__ == "0.36.2", "Διαφορετική έκδοση huggingface-hub"
assert metadata.version("accelerate") == "1.10.1", "Διαφορετική έκδοση accelerate"
print(json.dumps({
    "python": __import__("platform").python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "huggingface_hub": huggingface_hub.__version__,
    "accelerate": metadata.version("accelerate"),
    "gpu": torch.cuda.get_device_name(0),
}))
"""

environment_json = setup_command(
    [sys.executable, "-c", textwrap.dedent(environment_code)], cwd=REPO_DIR, capture=True
)
environment_info = json.loads(environment_json)
for name, value in environment_info.items():
    print(f"{name}: {value}")
GPU_READY = True

## 5. Σύγκριση C έναντι E μόνο στα development cases

Το script επαληθεύει ξανά τη συμβατότητα του προηγούμενου run. Επαναχρησιμοποιεί τα
upstream bytes και το προηγούμενο model revision. Δεν κάνει κλήσεις στο Jev ούτε
φορτώνει DistilBERT. Φόρτωση/warmup και χρόνος κάθε κριτικής καταγράφονται χωριστά.

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

if not REFERENCE_READY:
    raise RuntimeError("Πρώτα ανέβασε και έλεγξε το προηγούμενο ZIP.")
if not GPU_READY:
    raise RuntimeError("Πρώτα ολοκλήρωσε τον έλεγχο GPU.")
if file_sha256(SCRIPT_PATH) != SCRIPT_SHA256 or file_sha256(REVIEWS_PATH) != REVIEWS_SHA256:
    raise RuntimeError("Το script ή τα δεδομένα άλλαξαν.")
run_tag = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
RUN_DIR = REPO_DIR / "runs" / ("generator_prompt_v3_dev_" + run_tag)
RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
LOG_PATH = RUN_DIR.parent / (RUN_DIR.name + "_execution.txt")
command = [sys.executable, "-u", "scripts/compare_triage_generators.py", "dev",
           "--candidates", "C", "E", "--reference-run", str(REFERENCE_DIR),
           "--output", str(RUN_DIR)]
process_env = {name: value for name, value in os.environ.items() if name != SECRET_NAME}
process_env["TOKENIZERS_PARALLELISM"] = "false"
print("Run:", RUN_DIR.name)
with LOG_PATH.open("w", encoding="utf-8") as log:
    with subprocess.Popen(command, cwd=REPO_DIR, env=process_env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            if os.environ.get(SECRET_NAME):
                line = line.replace(os.environ[SECRET_NAME], "[REDACTED]")
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        exit_code = process.wait()
del process_env
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH.replace(RUN_DIR / "execution.txt")
print("Exit:", exit_code)
print("Συνέχισε για έλεγχο και κατέβασμα της αναφοράς, ακόμη και αν υπήρξε σφάλμα.")

## 6. Πληρότητα και σύνοψη

Ελέγχει 24 IDs για κάθε prompt, ίδιες upstream προβλέψεις, ίδιο Qwen revision και
μηδενικές νέες κλήσεις API. Ξεχωρίζει το πλήρες JSON χωρίς διπλά κλειδιά από την
αποδοχή αποσπασμάτων του parser. Κανένας από αυτούς τους ελέγχους δεν αποδεικνύει
ότι μια πρόταση είναι σωστή ή χρήσιμη.

In [ ]:
from collections import Counter
from IPython.display import Markdown, display

issues = []
required = ("run.json", "results.jsonl", "upstream.json", "human_review.csv", "annotation_key.json", "summary.md", "reference_run.json")
missing = [name for name in required if not (RUN_DIR / name).is_file()]
if exit_code != 0:
    issues.append(f"Exit code {exit_code}.")
if missing:
    issues.append("Λείπουν αρχεία: " + ", ".join(missing))
else:
    info = json.loads((RUN_DIR / "run.json").read_text())
    records = [json.loads(line) for line in (RUN_DIR / "results.jsonl").read_text().splitlines()]
    upstream = json.loads((RUN_DIR / "upstream.json").read_text())
    if not info["execution_complete"] or info["quality_evaluated"]:
        issues.append("Λανθασμένη κατάσταση εκτέλεσης ή ανθρώπινης αξιολόγησης.")
    if info["source_commit"] != SOURCE_COMMIT or info["dataset_sha256"] != REVIEWS_SHA256:
        issues.append("Διαφέρουν ο κώδικας ή τα δεδομένα.")
    if info["split"] != "dev" or set(info["candidates"]) != set(EXPECTED_CANDIDATES):
        issues.append("Δεν εκτελέστηκε η σωστή σύγκριση ανάπτυξης.")
    if not info.get("upstream_reused") or info["api_cost"]["attempts"] != 0:
        issues.append("Έγιναν νέες κλήσεις API ή δεν επαναχρησιμοποιήθηκε το upstream.")
    if (info.get("reference_run", {}).get("run_sha256") != REFERENCE_RUN_SHA256
            or file_sha256(RUN_DIR / "upstream.json") != REFERENCE_UPSTREAM_SHA256):
        issues.append("Διαφέρει η αναφορά ή το upstream από το προηγούμενο run.")
    if any(config.get("revision") != MODEL_REVISION for config in info["candidates"].values()):
        issues.append("Δεν χρησιμοποιήθηκαν τα προηγούμενα βάρη του Qwen.")
    if Counter(row["id"] for row in upstream["records"]) != Counter(EXPECTED_IDS):
        issues.append("Το upstream δεν καλύπτει τα 24 IDs μία φορά.")
    if any(row["complaints"]["status"] != "ok" for row in upstream["records"]):
        issues.append("Αποτυχία Jev.")
    stage_hashes = {}
    for candidate in EXPECTED_CANDIDATES:
        subset = [row for row in records if row["candidate"] == candidate]
        if Counter(row["id"] for row in subset) != Counter(EXPECTED_IDS):
            issues.append(candidate + ": λείπουν ή επαναλαμβάνονται IDs.")
        if any(row["error"] for row in subset):
            issues.append(candidate + ": σφάλμα παραγωγής.")
        for row in subset:
            previous = stage_hashes.setdefault(row["id"], row["upstream_sha256"])
            if previous != row["upstream_sha256"]:
                issues.append("Δεν χρησιμοποιήθηκαν ίδιες προβλέψεις upstream.")
    for name, expected in info["files"].items():
        if file_sha256(RUN_DIR / name) != expected:
            issues.append("Διαφέρει το fingerprint του " + name)
    display(Markdown((RUN_DIR / "summary.md").read_text()))
    display(Markdown("**Δεν επιλέχθηκε νικητής.** Συμπλήρωσε ανθρώπινα το human_review.csv με το πρωτόκολλο."))
checks = {"source_commit": SOURCE_COMMIT, "process_exit_code": exit_code,
          "issues": list(dict.fromkeys(issues)), "environment": environment_info,
          "human_quality_evaluated": False}
(RUN_DIR / "notebook_checks.json").write_text(json.dumps(checks, ensure_ascii=False, indent=2) + "\n")
print("Έλεγχοι εκτέλεσης:", "OK" if not issues else "; ".join(issues))

## 7. Κατέβασμα

Το ZIP περιλαμβάνει ακατέργαστες απαντήσεις, αποδεκτά μέτρα, revisions, fingerprints,
χρόνους, διαθέσιμη χρήση/κόστος Jev, log και κενό φύλλο ανθρώπινης αξιολόγησης.
Αποθηκεύεται και σε αποτυχημένο run για διάγνωση. Δώσε στον αξιολογητή μόνο το
human_review.csv· κράτησε χωριστά το annotation_key.json με τις ταυτότητες των παραλλαγών.

In [ ]:
import zipfile
from google.colab import files

export_paths = sorted(path for path in RUN_DIR.iterdir() if path.is_file())
_secret_for_check = os.environ.get(SECRET_NAME, "")
if _secret_for_check and any(_secret_for_check.encode() in path.read_bytes() for path in export_paths):
    del _secret_for_check
    raise RuntimeError("Δεν εξάγεται αναφορά που περιέχει Secret.")
del _secret_for_check
ZIP_PATH = RUN_DIR.parent / (RUN_DIR.name + ".zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in export_paths:
        archive.write(path, arcname=path.name)
print("ZIP:", ZIP_PATH.name)
files.download(str(ZIP_PATH))
if issues:
    raise RuntimeError("Το ZIP κατέβηκε, αλλά η εκτέλεση έχει προβλήματα. Δες το execution.txt.")

## Μετά την εκτέλεση

Δώσε στον αξιολογητή μόνο το `human_review.csv` με τις 48 εξόδους και κράτησε χωριστά
το `annotation_key.json`. Έλεγξε πραγματικά εκκρεμή προβλήματα, σωστά αποσπάσματα,
χρήσιμα μέτρα για τη διοίκηση και παράπονα που παραλείφθηκαν.

Το παλιό API κόστος διατηρείται στην αναφορά ως **ιστορικό**, χωρίς να χρεώνεται ξανά
στη νέα σύγκριση. Η τοπική GPU χρέωση μένει άγνωστη.
Δεν έχει επιλεγεί νικητής και δεν γίνεται αυτόματα freeze ή εκτέλεση του reserved set.
Μετά την ανθρώπινη αξιολόγηση, η επιλογή γίνεται με το υπάρχον πρωτόκολλο του repo.